# Silver - CFPB API Cleaning

## Setup

In [1]:
from pyspark.sql.functions import col, trim, upper, lower, substring, when, lit, to_date, current_timestamp, concat

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 3, Finished, Available, Finished, False)

In [2]:
bronze_api_table = "bronze.cfpb_complaints_api"
target_silver_table = "silver.complaints"

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 4, Finished, Available, Finished, False)

## Read Bronze

In [3]:
bronze_df = (
    spark.table(bronze_api_table)
    .dropDuplicates(["complaint_id"])
)

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 5, Finished, Available, Finished, False)

## Transformations

In [4]:
standardized_df = (
    bronze_df
    .select(
        col("date_received"),
        col("product"),
        col("sub_product"),
        col("issue"),
        col("sub_issue"),
        col("complaint_what_happened").alias("consumer_complaint_narrative"),
        col("company_public_response"),
        col("company"),
        col("state"),
        col("zip_code"),
        col("tags"),
        col("submitted_via"),
        col("date_sent_to_company"),
        col("company_response").alias("company_response_to_consumer"),
        col("timely").alias("timely_response"),
        col("complaint_id"),
        col("bronze_ingestion_timestamp").alias("ingestion_timestamp"),
        lit("cfpb_api").alias("source_file"),
        concat(lit("api_"), col("api_start_date"), lit("_"), col("api_end_date")).alias("batch_id")
    )
)

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 6, Finished, Available, Finished, False)

In [5]:
silver_dates_df = (
    standardized_df
    .withColumn(
        "date_received_clean",
        to_date(col("date_received"))
    )
    .withColumn(
        "date_sent_to_company_clean",
        to_date(col("date_sent_to_company"))
    )
)

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 7, Finished, Available, Finished, False)

In [6]:
zip_text = trim(col("zip_code"))

missing_zip = (
    col("zip_code").isNull()
    | (zip_text == "")
    | (lower(zip_text) == "unknown")
)

valid_zip = zip_text.rlike("^[0-9]{5}")

silver_location_df = (
    silver_dates_df
    .withColumn(
        "state_clean",
        when(
            col("state").isNull() | (trim(col("state")) == ""),
            lit(None)
        ).otherwise(upper(trim(col("state"))))
    )
    .withColumn(
        "zip_code_clean",
        when(missing_zip, lit(None))
        .when(valid_zip, substring(zip_text, 1, 5))
        .otherwise(lit(None))
    )
    .withColumn(
        "zip_code_status",
        when(missing_zip, "missing_zip")
        .when(valid_zip, "valid_zip")
        .otherwise("masked_zip")
    )
)

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 8, Finished, Available, Finished, False)

In [7]:
silver_complaints_df = (
    silver_location_df
    .withColumn("silver_processed_timestamp", current_timestamp())
)

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 9, Finished, Available, Finished, False)

## Merge Into Silver Table

In [8]:
silver_complaints_df.createOrReplaceTempView("cfpb_api_silver_updates")

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 10, Finished, Available, Finished, False)

In [9]:
spark.sql(f"""
    MERGE INTO {target_silver_table} AS target
    USING cfpb_api_silver_updates AS source
        ON target.complaint_id = source.complaint_id

    WHEN MATCHED THEN
        UPDATE SET *

    WHEN NOT MATCHED THEN
        INSERT *
""")

StatementMeta(, 8f4991a7-9f7a-4127-8341-5daa9488b138, 11, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]